In [2]:
import os
import json
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 配置参数
population_file = 'data/output/population_points.json'  # 你可以修改为你实际的文件路径

# 检查文件是否存在
if not os.path.exists(population_file):
    print(f"❌ 找不到人口数据文件: {population_file}")
else:
    try:
        # 读取JSON文件
        with open(population_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # 提取学生代理数据
        agents = data['agents']['student_agent']['states']['default']['agents']
        num_student_agents = len(agents)
        print(f"找到 {num_student_agents} 个学生代理个体")
        
        
    except Exception as e:
        print(f"❌ 处理文件时出错: {str(e)}")

找到 1914 个学生代理个体


In [ ]:
agent_data=agents[3]['variables']
agent_data

In [ ]:
agent_data['x'] 


256.53206457581945

In [1]:
from pyflamegpu import *
import pyflamegpu.codegen
import sys

In [6]:
?SimulationConfig()

Object `SimulationConfig()` not found.


In [1]:
AGENT_COUNT = 16384
ENV_WIDTH = int(AGENT_COUNT**(1/3))

In [3]:
stairwell_file = 'data/output/transformed_stairwell.geojson'

with open(stairwell_file, 'r', encoding='utf-8') as f:
    stairwell_data = json.load(f)

# 提取楼梯间特征
stairwell_features = stairwell_data['features']
num_stairwell_agents = len(stairwell_features)
print(f"初始化 {num_stairwell_agents} 个楼梯间代理")
    


初始化 7 个楼梯间代理


In [6]:
feature = stairwell_features[1]
coordinates = feature['geometry']['coordinates']
coordinates[0]

320.31274575542193

In [3]:
config_path = os.path.join(os.path.dirname(__file__), '../config/env.yaml')
if os.path.exists(config_path):
    with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    base_population = config.get('base_population', 50)

NameError: name '__file__' is not defined

In [2]:
from pyflamegpu import *

pyflamegpu.clearRTCDiskCache()

In [14]:
import numpy as np
import json
from shapely.geometry import Point

def generate_attraction_matrix(
    m, n,
    attraction_points,  # 手动输入的高吸引力点列表 [(x, y, radius, attraction), ...]
    normalize=False,     # 是否归一化到 [0, 1]
    output_json=False    # 是否返回 JSON 格式
):
    """
    Generate an attraction matrix based on specified attraction points with radial influence.
    
    Parameters:
    - m, n: Dimensions of the output matrix (rows, columns)
    - attraction_points: List of tuples (x, y, radius, attraction_value)
    - normalize: Whether to normalize the matrix to [0, 1]
    - output_json: Whether to return the result as JSON
    
    Returns:
    - Either a numpy array or JSON string representing the attraction matrix
    """
    
    # 1. 初始化总体边界矩阵
    attraction_matrix = np.zeros((m, n))
    
    # 2. 计算每个网格点的吸引力
    for x, y, radius, attraction in attraction_points:
        # 确保坐标在矩阵范围内
        x = max(0, min(m-1, x))
        y = max(0, min(n-1, y))
        
        # 创建网格坐标
        rows, cols = np.indices((m, n))
        
        # 计算每个点到吸引力中心的距离
        distances = np.sqrt((rows - x)**2 + (cols - y)**2)
        
        # 应用高斯衰减函数 (在半径范围内)
        decay = np.exp(-(distances**2) / (2 * (radius/3)**2))  # radius/3 makes it fall to ~0.1 at radius
        influence = attraction * np.where(distances <= radius, decay, 0)
        
        # 叠加到总体矩阵
        attraction_matrix += influence
    
    # 3. 归一化（可选）
    if normalize:
        max_val = np.max(attraction_matrix)
        if max_val > 0:
            attraction_matrix = attraction_matrix / max_val
    
    # 4. 返回 JSON 或矩阵
    if output_json:
        json_data = {
            "macro_environment": {
                "map": attraction_matrix.flatten().tolist()  # 展平为 1D 数组
            }
        }
        return json.dumps(json_data, indent=2)
    else:
        return attraction_matrix

In [15]:
m, n = 20,20  # 5x5 网格

# 手动输入高吸引力点 (x, y, radius, attraction)
attraction_points = [
    (10, 10, 4, 100),  # 中心点 (2,2)，半径 2，吸引力 100
    (0, 4, 4, 80),  # 点 (0,4)，半径 1.5，吸引力 80
]

# 生成 JSON 格式
output = generate_attraction_matrix(m, n, attraction_points)

output

array([[  0.88871972,   6.3647607 ,  25.97219739,  60.38716816,
         80.        ,  60.38716816,  25.97219739,   6.3647607 ,
          0.88871972,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ],
       [  0.        ,   4.80437343,  19.60484314,  45.58262598,
         60.38716816,  45.58262598,  19.60484314,   4.80437343,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ],
       [  0.        ,   2.06633526,   8.43193796,  19.60484314,
         25.97219739,  19.60484314,   8.43193796,   2.06633526,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ],
       [  0.        ,   0.        ,  

In [12]:
import numpy as np
from shapely.geometry import Point

def generate_attraction_matrix(
    m, n,
    attraction_points,  # 手动输入的高吸引力点列表 [(x, y, radius, attraction), ...]
    normalize=False,     # 是否归一化到 [0, 1]
    output_json=False   # 是否返回 JSON 格式
):
    """
    生成一个 m×n 的吸引力矩阵，基础值为1，高吸引力点在其周边衰减叠加。
    
    参数:
        m (int): 网格行数
        n (int): 网格列数
        attraction_points (list): 高吸引力点列表，格式 [(x, y, radius, attraction), ...]
        normalize (bool): 是否归一化到 [0, 1]（默认 True）
        output_json (bool): 是否返回 JSON 格式（默认 False）
    
    返回:
        dict: 如果 output_json=True，返回 JSON 结构
        np.ndarray: 如果 output_json=False，返回吸引力矩阵
    """
    # 1. 初始化吸引力矩阵（基础值 = 1）
    attraction_matrix = np.ones((m, n))
    
    # 2. 计算高吸引力点的影响（叠加到基础值上）
    for i in range(m):
        for j in range(n):
            point = Point(j + 0.5, i + 0.5)  # 单元格中心点坐标
            
            # 计算所有高吸引力点的影响
            for x, y, radius, attraction in attraction_points:
                dist = np.sqrt((point.x - x) ** 2 + (point.y - y) ** 2)
                if dist <= radius:
                    # 高斯衰减
                    gaussian_factor = np.exp(-(dist ** 2) / (2 * (radius / 2) ** 2))
                    attraction_matrix[i, j] += attraction * gaussian_factor
    
    # 3. 归一化（可选）
    if normalize:
        max_val = np.max(attraction_matrix)
        if max_val > 0:
            attraction_matrix = attraction_matrix / max_val
    
    # 4. 返回 JSON 或矩阵
    if output_json:
        json_data = {
            "macro_environment": {
                "map": attraction_matrix.flatten().tolist()  # 展平为 1D 数组
            }
        }
        return json_data
    else:
        return attraction_matrix



In [13]:

m, n = 20, 18  # 5x5 网格

# 手动输入高吸引力点 (x, y, radius, attraction)
attraction_points = [
    (10, 10, 2, 5),  # 中心点 (2,2)，半径 2，额外吸引力 5
    (0, 4, 1.5, 3), # 点 (0,4)，半径 1.5，额外吸引力 3
]

# 生成矩阵
matrix = generate_attraction_matrix(m, n, attraction_points, normalize=True)

matrix

array([[0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167],
       [0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167],
       [0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167],
       [0.59737205, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167, 0.20433167, 0.20433167,
        0.20433167, 0.20433167, 0.20433167],
    

In [23]:
def generate_attraction_matrix(
    m, n,
    attraction_points,  # 手动输入的高吸引力点列表 [(x, y, radius, attraction), ...]
    normalize=False,     # 是否归一化到 [0, 1]
    output_json=False    # 是否返回 JSON 格式
):
    """
    Generate an attraction matrix based on specified attraction points with radial influence.
    
    Parameters:
    - m, n: Dimensions of the output matrix (rows, columns)
    - attraction_points: List of tuples (x, y, radius, attraction_value)
    - normalize: Whether to normalize the matrix to [0, 1]
    - output_json: Whether to return the result as JSON
    
    Returns:
    - Either a numpy array or JSON string representing the attraction matrix
    """
    
    # 1. 初始化总体边界矩阵
    attraction_matrix = np.zeros((m, n))
    
    # 2. 计算每个网格点的吸引力
    for x, y, radius, attraction in attraction_points:
        # 确保坐标在矩阵范围内
        x = max(0, min(m-1, x))
        y = max(0, min(n-1, y))
        
        # 创建网格坐标
        rows, cols = np.indices((m, n))
        
        # 计算每个点到吸引力中心的距离
        distances = np.sqrt((rows - x)**2 + (cols - y)**2)
        
        # 应用高斯衰减函数 (在半径范围内)
        decay = np.exp(-(distances**2) / (2 * (radius/2)**2))  # radius/3 makes it fall to ~0.1 at radius
        influence = attraction * np.where(distances <= radius, decay, 0)
        
        # 叠加到总体矩阵
        attraction_matrix += influence
    
    # 3. 归一化（可选）
    if normalize:
        max_val = np.max(attraction_matrix)
        if max_val > 0:
            attraction_matrix = attraction_matrix / max_val
    
    # 4. 返回 JSON 或矩阵
    if output_json:
        json_data = {
            "macro_environment": {
                "map": attraction_matrix.flatten().tolist()  # 展平为 1D 数组
            }
        }
        return json.dumps(json_data, indent=2)
    else:
        return attraction_matrix

In [24]:
m, n = 20, 18  # 5x5 网格

# 手动输入高吸引力点 (x, y, radius, attraction)
attraction_points = [
    (10, 10, 2, 100),  # 中心点 (2,2)，半径 2，额外吸引力 5
    (0, 4, 1.5, 80), # 点 (0,4)，半径 1.5，额外吸引力 3
]

# 生成矩阵
matrix = generate_attraction_matrix(m, n, attraction_points, normalize=False)

matrix

array([[  0.        ,   0.        ,   0.        ,  32.88898324,
         80.        ,  32.88898324,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ],
       [  0.        ,   0.        ,   0.        ,  13.52106523,
         32.88898324,  13.52106523,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ],
       [  0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ],
       [  0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   0.        ,   0.   

第二种处理方法：

In [33]:
import numpy as np
from collections import deque

def generate_attraction_matrix_direct(
    m, n,
    attraction_points,  # [(x, y, radius, base_attraction)]
    decay_rate=0.5,     # 每层衰减比例 (0~1)
    normalize=False,
    output_json=True
):
    """
    通过广度优先搜索 (BFS) 实现逐层衰减的吸引力矩阵
    """
    attraction_matrix = np.zeros((m, n))
    
    for x, y, radius, base_attraction in attraction_points:
        x, y = int(np.clip(x, 0, m-1)), int(np.clip(y, 0, n-1))
        visited = np.zeros((m, n), dtype=bool)
        queue = deque()
        
        # 初始化中心点
        queue.append((x, y, base_attraction))
        visited[x, y] = True
        attraction_matrix[x, y] += base_attraction
        
        # BFS 向外扩散
        while queue:
            cx, cy, current_attraction = queue.popleft()
            
            # 遍历四个邻居方向
            for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                nx, ny = cx + dx, cy + dy
                
                # 检查边界和是否已访问
                if 0 <= nx < m and 0 <= ny < n and not visited[nx, ny]:
                    distance = abs(nx - x) + abs(ny - y)  # 曼哈顿距离
                    if distance <= radius:
                        decayed_attraction = base_attraction * (decay_rate ** distance)
                        attraction_matrix[nx, ny] += decayed_attraction
                        visited[nx, ny] = True
                        queue.append((nx, ny, decayed_attraction))
    
    if normalize:
        max_val = np.max(attraction_matrix)
        if max_val > 0:
            attraction_matrix /= max_val
    
    if output_json:
        return json.dumps({"macro_environment": {"map": attraction_matrix.flatten().tolist()}})
    else:
        return attraction_matrix

In [35]:
m, n = 20, 18  # 5x5 网格

# 手动输入高吸引力点 (x, y, radius, attraction)
attraction_points = [
    (10, 10, 2, 100),  # 中心点 (2,2)，半径 2，额外吸引力 5
    (0, 4, 2, 80), # 点 (0,4)，半径 1.5，额外吸引力 3
]

# 生成矩阵
matrix = generate_attraction_matrix_direct(m, n, attraction_points, normalize=False)



In [39]:
def ccw(A, B, C):
    """判断点C是否在向量AB的逆时针方向（用于判断线段相交）"""
    return (C[1] - A[1]) * (B[0] - A[0]) - (B[1] - A[1]) * (C[0] - A[0])

def intersect(A, B, C, D):
    """判断线段AB和线段CD是否相交"""
    return (ccw(A, C, D) * ccw(B, C, D) < 0) and (ccw(A, B, C) * ccw(A, B, D) < 0)

def point_in_polygon(p, polygon):
    """判断点p是否在多边形polygon内"""
    x, y = p
    n = len(polygon)
    inside = False
    for i in range(n):
        x1, y1 = polygon[i]
        x2, y2 = polygon[(i + 1) % n]
        if y > min(y1, y2):
            if y <= max(y1, y2):
                if x <= max(x1, x2):
                    if y1 != y2:
                        x_intersect = (y - y1) * (x2 - x1) / (y2 - y1) + x1
                    if x1 == x2 or x <= x_intersect:
                        inside = not inside
    return inside

def line_intersects_polygon(line, polygon):
    """判断线段line是否与多边形polygon相交"""
    A, B = line
    n = len(polygon)
    # 检查线段的端点是否在多边形内
    if point_in_polygon(A, polygon) or point_in_polygon(B, polygon):
        return True
    # 检查线段是否与多边形的任意一条边相交
    for i in range(n):
        C = polygon[i]
        D = polygon[(i + 1) % n]
        if intersect(A, B, C, D):
            return True
    return False

# 示例用法
polygon = [(1, 1), (1, 4), (4, 4), (4, 1)]  # 一个正方形多边形
line = [(0, 2), (5, 2)]  # 一条水平线段
print(line_intersects_polygon(line, polygon))  # 输出: True


True


In [1]:
import math
from typing import List, Tuple, Optional

class AStarAlgorithm:
    def __init__(self, vertices: List[Tuple[float, float, float]], edges: List[Tuple[int, int]]):
        self.vertices = vertices
        self.edges = edges
        self.open_list = []
        self.parent = []
        self.g_score = []
        self.h_score = []
        self.f_score = []
        
    def compute_euclidean_distance(self, v1: int, v2: int) -> float:
        """计算两个顶点之间的欧几里得距离"""
        x1, y1, z1 = self.vertices[v1]
        x2, y2, z2 = self.vertices[v2]
        return math.sqrt((x2 - x1)**2 + (y2 - y1)**2 + (z2 - z1)**2)
    
    def find_lowest_f_score_vertex(self) -> int:
        """在open列表中找到f分数最低的顶点"""
        lowest_f_score = float('inf')
        lowest_score_vertex = -1
        
        for v in self.open_list:
            if self.f_score[v] < lowest_f_score:
                lowest_f_score = self.f_score[v]
                lowest_score_vertex = v
                
        return lowest_score_vertex
    
    def find_neighbors(self, v: int) -> List[int]:
        """找到给定顶点的所有邻居顶点"""
        neighbors = []
        
        for edge in self.edges:
            v1, v2 = edge
            if v == v1:
                neighbors.append(v2)
            elif v == v2:
                neighbors.append(v1)
                
        return neighbors
    
    def construct_path(self, current: int) -> List[int]:
        """从终点回溯构建路径"""
        path = [current]
        
        while self.parent[current] != -1:
            current = self.parent[current]
            path.append(current)
            
        path.reverse()
        return path
    
    def a_star(self, start_vertex: int, destination_vertex: int) -> Optional[List[int]]:
        """执行A*算法寻找最短路径"""
        # 初始化数据结构
        self.open_list = []
        self.parent = [-1] * len(self.vertices)
        self.g_score = [float('inf')] * len(self.vertices)
        self.h_score = [self.compute_euclidean_distance(v, destination_vertex) for v in range(len(self.vertices))]
        self.f_score = [float('inf')] * len(self.vertices)
        
        # 设置起点
        self.open_list.append(start_vertex)
        self.g_score[start_vertex] = 0
        self.f_score[start_vertex] = self.h_score[start_vertex]
        
        while self.open_list:
            current_vertex = self.find_lowest_f_score_vertex()
            
            if current_vertex == destination_vertex:
                return self.construct_path(current_vertex)
            
            self.open_list.remove(current_vertex)
            neighbors = self.find_neighbors(current_vertex)
            
            for neighbor in neighbors:
                tentative_g_score = self.g_score[current_vertex] + self.compute_euclidean_distance(current_vertex, neighbor)
                
                if tentative_g_score < self.g_score[neighbor]:
                    self.parent[neighbor] = current_vertex
                    self.g_score[neighbor] = tentative_g_score
                    self.f_score[neighbor] = self.g_score[neighbor] + self.h_score[neighbor]
                    
                    if neighbor not in self.open_list:
                        self.open_list.append(neighbor)
        
        return None  # 没有找到路径


# 使用示例
if __name__ == "__main__":
    # 定义顶点和边
    vertices = [
        (0, 0, 0),  # 0
        (1, 0, 0),  # 1
        (1, 1, 0),  # 2
        (0, 1, 0),  # 3
        (0.5, 1.5, 0)  # 4
    ]
    
    edges = [
        (0, 1),  # 边0-1
        (1, 2),  # 边1-2
        (2, 3),  # 边2-3
        (3, 0),  # 边3-0
        (2, 4),  # 边2-4
        (3, 4)   # 边3-4
    ]
    
    # 创建A*实例并查找路径
    astar = AStarAlgorithm(vertices, edges)
    path = astar.a_star(0, 4)  # 从顶点0到顶点4
    
    if path:
        print("找到路径:", path)
        print("路径顶点坐标:")
        for v in path:
            print(f"顶点{v}: {vertices[v]}")
    else:
        print("没有找到路径")

找到路径: [0, 3, 4]
路径顶点坐标:
顶点0: (0, 0, 0)
顶点3: (0, 1, 0)
顶点4: (0.5, 1.5, 0)
